# 02 — Synthetic Primary Analysis (DR-RF)

Generates primary manuscript figures and tables using **DR-RF only**.

Sections:
1. Load results for all alpha values
2. DR-RF scissors chart — ATE MSE vs alpha (5 methods, 95% CI bands)
3. DR-RF scissors chart — CATE MSE vs alpha
4. Paired-difference table per alpha
5. Ranking metrics (alpha > 0)
6. Summary table for paper

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'synthetic'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    METHODS_ORDER, LEARNERS_ORDER, paired_ci, ranking_comparison_table, format_ci,
    resolve_alternative, describe_paired_test, build_paper_summary_table_by_alpha, significance_stars
)
from helpers.plotting import plot_scissors_chart, METHOD_LABELS

os.makedirs(CFG.PLOTS_DIR, exist_ok=True)
print(f'Setup complete. CI_METHOD={CFG.CI_METHOD!r}, CI_SIDED={CFG.CI_SIDED!r}')


## 1. Load Results for All Alpha Values

In [ ]:
results_by_alpha = {}
for alpha in CFG.ALPHA_VALUES:
    full_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    debug_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    if os.path.exists(full_path):
        path = full_path
    else:
        path = debug_path
        print(f'WARNING: full-scale checkpoint not found for alpha={alpha:.2f}; '
              f'falling back to DEBUG checkpoint (small N_TRAIN/N_TEST, few seeds). '
              f'Run the full loop in 01_experiment.ipynb for real results.')
    results_by_alpha[alpha] = load_checkpoint(path)
    print(f'alpha={alpha:.2f}: {len(results_by_alpha[alpha])} seeds  [{os.path.basename(path)}]')


In [ ]:
from helpers.plotting import plot_scissors_chart_combined

fig = plot_scissors_chart_combined(
    results_by_alpha,
    panels=[
        {'metric': 'cate_mse', 'alpha_values': CFG.ALPHA_VALUES,
         'title': f'CATE MSE vs α ({CFG.PRIMARY_LEARNER})'},
        {'metric': 'kendall_tau', 'alpha_values': [a for a in CFG.ALPHA_VALUES if a > 0],
         'title': f'Kendall τ vs α ({CFG.PRIMARY_LEARNER})'},
    ],
    learner=CFG.PRIMARY_LEARNER,
    save_path=os.path.join(CFG.PLOTS_DIR, 'combined_cate_kendall.png'),
)
plt.show()


## 2. DR-RF Scissors Chart — ATE MSE vs Alpha

In [ ]:
fig = plot_scissors_chart(
    results_by_alpha,
    alpha_values=CFG.ALPHA_VALUES,
    learner=CFG.PRIMARY_LEARNER,
    metric='ate_mse',
    title=f'ATE MSE vs Heterogeneity — {CFG.PRIMARY_LEARNER}',
    save_path=os.path.join(CFG.PLOTS_DIR, 'scissors_ate_mse.png'),
)
plt.show()

## 3. DR-RF Scissors Chart — CATE MSE vs Alpha

In [ ]:
fig = plot_scissors_chart(
    results_by_alpha,
    alpha_values=CFG.ALPHA_VALUES,
    learner=CFG.PRIMARY_LEARNER,
    metric='cate_mse',
    title=f'CATE MSE vs Heterogeneity — {CFG.PRIMARY_LEARNER}',
    save_path=os.path.join(CFG.PLOTS_DIR, 'scissors_cate_mse.png'),
)
plt.show()

## 4. Paired Differences per Alpha (CDV_SEPARATE vs others)

In [ ]:
alt_ate = resolve_alternative(CFG.CI_SIDED, lower_is_better=True)
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = np.array([
        sr['metrics'].get('CDV_SEPARATE', {}).get(CFG.PRIMARY_LEARNER, {}).get('ate_mse', np.nan)
        for sr in res.values()
    ])
    for method in METHODS_ORDER:
        if method == 'CDV_SEPARATE':
            continue
        other_vals = np.array([
            sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('ate_mse', np.nan)
            for sr in res.values()
        ])
        ci = paired_ci(cdv_vals, other_vals, method=CFG.CI_METHOD, alternative=alt_ate)
        rows.append({
            'alpha': alpha,
            'vs_method': METHOD_LABELS.get(method, method),
            'mean_delta': ci['mean'],
            'paired 95% CI of Δ (CDV_SEPARATE − method)': format_ci(ci['ci_lo'], ci['ci_hi'], decimals=5),
            'p_value (paired)': ci['p_value'],
            'n_seeds': ci['n'],
        })

diff_df = pd.DataFrame(rows)
display(diff_df.round(5))
diff_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paired_differences.csv'), index=False)


## 4b. Paired Differences per Alpha — CATE (CDV_SEPARATE vs others)

In [ ]:
alt_cate = resolve_alternative(CFG.CI_SIDED, lower_is_better=True)
print(describe_paired_test('CDV_SEPARATE', 'method', 'CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

rows_cate = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = np.array([
        sr['metrics'].get('CDV_SEPARATE', {}).get(CFG.PRIMARY_LEARNER, {}).get('cate_mse', np.nan)
        for sr in res.values()
    ])
    for method in METHODS_ORDER:
        if method == 'CDV_SEPARATE':
            continue
        other_vals = np.array([
            sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('cate_mse', np.nan)
            for sr in res.values()
        ])
        ci = paired_ci(cdv_vals, other_vals, method=CFG.CI_METHOD, alternative=alt_cate)
        rows_cate.append({
            'alpha': alpha,
            'vs_method': METHOD_LABELS.get(method, method),
            'mean_delta': ci['mean'],
            'paired 95% CI of Δ (CDV_SEPARATE − method)': format_ci(ci['ci_lo'], ci['ci_hi'], decimals=5),
            'p_value (paired)': ci['p_value'],
            'n_seeds': ci['n'],
        })

diff_cate_df = pd.DataFrame(rows_cate)
display(diff_cate_df.round(5))
diff_cate_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paired_differences_cate.csv'), index=False)


## 4c. DR-RF Scissors Chart — Kendall Tau vs Alpha

In [ ]:
fig = plot_scissors_chart(
    results_by_alpha,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a>0],
    learner=CFG.PRIMARY_LEARNER,
    metric='kendall_tau',
    title=f'Kendall Tau vs Heterogeneity — {CFG.PRIMARY_LEARNER}',
    save_path=os.path.join(CFG.PLOTS_DIR, 'scissors_kendall_tau.png'),
)
plt.show()

## 4d. DR-RF Scissors Chart — Spearman Rho vs Alpha

In [ ]:
fig = plot_scissors_chart(
    results_by_alpha,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a>0],
    learner=CFG.PRIMARY_LEARNER,
    metric='spearman_rho',
    title=f'Spearman Rho vs Heterogeneity — {CFG.PRIMARY_LEARNER}',
    save_path=os.path.join(CFG.PLOTS_DIR, 'scissors_spearman_rho.png'),
)
plt.show()

## 5. Ranking Metrics (alpha > 0)

In [ ]:
print(f'Ranking Metrics ({CFG.PRIMARY_LEARNER}), alpha > 0 only')
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
rank_rows = []
for alpha in [a for a in CFG.ALPHA_VALUES if a > 0]:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        tau_vals = [sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('kendall_tau', np.nan)
                    for sr in res.values()]
        rho_vals = [sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('spearman_rho', np.nan)
                    for sr in res.values()]
        rank_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'kendall_tau_mean': np.nanmean(tau_vals), 'kendall_tau_std': np.nanstd(tau_vals),
            'spearman_rho_mean': np.nanmean(rho_vals), 'spearman_rho_std': np.nanstd(rho_vals),
        })

rank_df = pd.DataFrame(rank_rows)

# Paired 95% CI of Δ (CDV_SEPARATE − method), paired p-value, and improvement % vs CDV_SEPARATE; NaN for the CDV_SEPARATE row itself
cmp_rows = []
for alpha in [a for a in CFG.ALPHA_VALUES if a > 0]:
    res = results_by_alpha.get(alpha, {})
    tau_cmp = ranking_comparison_table(res, CFG.PRIMARY_LEARNER, metric='kendall_tau',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    rho_cmp = ranking_comparison_table(res, CFG.PRIMARY_LEARNER, metric='spearman_rho',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    for method in tau_cmp.index:
        cmp_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'kendall_tau_paired_95%_CI': tau_cmp.loc[method, 'paired 95% CI of Δ (CDV_SEPARATE − method)'],
            'kendall_tau_p_value_paired': tau_cmp.loc[method, 'p_value (paired)'],
            'kendall_tau_improvement_pct': tau_cmp.loc[method, 'improvement_pct'],
            'spearman_rho_paired_95%_CI': rho_cmp.loc[method, 'paired 95% CI of Δ (CDV_SEPARATE − method)'],
            'spearman_rho_p_value_paired': rho_cmp.loc[method, 'p_value (paired)'],
            'spearman_rho_improvement_pct': rho_cmp.loc[method, 'improvement_pct'],
        })

rank_df = rank_df.merge(pd.DataFrame(cmp_rows), on=['alpha', 'method'], how='left')
display(rank_df.round(4))


## 6. Summary Table for Paper

In [ ]:
summary_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        ate_vals  = [sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('ate_mse', np.nan)  for sr in res.values()]
        cate_vals = [sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('cate_mse', np.nan) for sr in res.values()]
        summary_rows.append({
            'alpha': alpha, 'method': method,
            'ate_mse_mean': np.nanmean(ate_vals), 'ate_mse_std': np.nanstd(ate_vals),
            'cate_mse_mean': np.nanmean(cate_vals), 'cate_mse_std': np.nanstd(cate_vals),
            'n_seeds': int(np.sum(np.isfinite(ate_vals))),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(5))
summary_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'primary_summary.csv'), index=False)
print(f'Saved: {CFG.ARTIFACTS_DIR}/primary_summary.csv')

## 7. Compact Paper Summary Tables (all alpha values, with significance)

The full per-alpha breakdown (Sections 4–5) is too wide for the paper. These
compact tables put **alpha values as columns** instead: row index is
(Metric, Method); the CDV_SEPARATE row shows its own raw mean ± std per alpha
(reference); every other method's cell shows the paired
Δ = CDV_SEPARATE − method (mean) with significance stars from the same paired
test, e.g. `-0.938***` (see `CFG.CI_METHOD` / `CFG.CI_SIDED`).
Significance: `*` p<0.1, `**` p<0.05, `***` p<0.01, `-` not significant / no data.


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

ate_cate_by_alpha = build_paper_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=CFG.ALPHA_VALUES,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
)
display(ate_cate_by_alpha)
ate_cate_by_alpha.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_ate_cate_by_alpha.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

rank_by_alpha = build_paper_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a > 0],
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
)
display(rank_by_alpha)
rank_by_alpha.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_kendall_spearman_by_alpha.csv'))


## 8. Compact Signed Summary Tables (metric as primary columns)

Two compact tables for the main text: rows = method (CDV_SEPARATE first,
reference); columns grouped first by **metric**, then by **alpha**, each with
two sub-columns: raw **Mean ± Std** and a **signed Δ%** vs CDV_SEPARATE
(positive always means CDV_SEPARATE is better, regardless of metric
direction) with significance stars from a two-sided paired test (see
`CFG.CI_METHOD`). Table 8a covers ATE/CATE MSE (all alpha); Table 8b covers
Kendall τ / Spearman ρ (alpha > 0 only, undefined at alpha=0).
Significance: `*` p<0.1, `**` p<0.05, `***` p<0.01, `-` no data.

In [ ]:
from helpers.metrics import build_signed_summary_table_by_alpha

print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided='two-sided', ci_method=CFG.CI_METHOD))

signed_ate_cate_by_alpha = build_signed_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=CFG.ALPHA_VALUES,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
)
display(signed_ate_cate_by_alpha)
signed_ate_cate_by_alpha.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_ate_cate_by_alpha.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided='two-sided', ci_method=CFG.CI_METHOD))

signed_rank_by_alpha = build_signed_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a > 0],
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
)
display(signed_rank_by_alpha)
signed_rank_by_alpha.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_kendall_spearman_by_alpha.csv'))


## 9. Multiplicity-Adjusted Summary Tables (FDR-BH correction)

Sections 7–8 each report a **raw, unadjusted** paired p-value per (metric,
method, alpha) cell — up to 4 non-reference methods × 6 alpha values per
metric, i.e. dozens of simultaneous pairwise comparisons — so the raw
significance stars there are optimistic (inflated false-positive rate under
multiple testing). The tables below repeat 7a/7b/8a/8b with p-values
adjusted for multiplicity using Benjamini-Hochberg (`adjust_method='fdr_bh'`,
controls the false discovery rate; see `helpers.metrics.adjust_pvalues`).
Each alpha is always its own family, never pooled across alpha; within an
alpha, ATE MSE, CATE MSE, Kendall τ, and Spearman ρ are each corrected as
their own separate family (`metric_families={}`) — 4 families per alpha.
Only the significance stars change here — means, CIs, and Δ% are identical
to the unadjusted tables above.

In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD)
      + '  [stars: FDR-BH adjusted, ATE MSE and CATE MSE each their own family, per alpha]')

ate_cate_by_alpha_fdr = build_paper_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=CFG.ALPHA_VALUES,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
    adjust_method='fdr_bh',
    metric_families={},
)
display(ate_cate_by_alpha_fdr)
ate_cate_by_alpha_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_ate_cate_by_alpha_fdr_bh.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD)
      + '  [stars: FDR-BH adjusted, Kendall τ and Spearman ρ each their own family, per alpha]')

rank_by_alpha_fdr = build_paper_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a > 0],
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
    adjust_method='fdr_bh',
    metric_families={},
)
display(rank_by_alpha_fdr)
rank_by_alpha_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_kendall_spearman_by_alpha_fdr_bh.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided='two-sided', ci_method=CFG.CI_METHOD)
      + '  [stars: FDR-BH adjusted, ATE MSE and CATE MSE each their own family, per alpha]')

signed_ate_cate_by_alpha_fdr = build_signed_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=CFG.ALPHA_VALUES,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
    adjust_method='fdr_bh',
    metric_families={},
)
display(signed_ate_cate_by_alpha_fdr)
signed_ate_cate_by_alpha_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_ate_cate_by_alpha_fdr_bh.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided='two-sided', ci_method=CFG.CI_METHOD)
      + '  [stars: FDR-BH adjusted, Kendall τ and Spearman ρ each their own family, per alpha]')

signed_rank_by_alpha_fdr = build_signed_summary_table_by_alpha(
    results_by_alpha,
    learner=CFG.PRIMARY_LEARNER,
    alpha_values=[a for a in CFG.ALPHA_VALUES if a > 0],
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
    adjust_method='fdr_bh',
    metric_families={},
)
display(signed_rank_by_alpha_fdr)
signed_rank_by_alpha_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_kendall_spearman_by_alpha_fdr_bh.csv'))


## 10. Combined Figure — CATE MSE & Kendall τ (side by side, shared legend)

Single figure with two panels sharing one legend, for the manuscript.


In [ ]:
from helpers.plotting import plot_scissors_chart_combined

fig = plot_scissors_chart_combined(
    results_by_alpha,
    panels=[
        {'metric': 'cate_mse', 'alpha_values': CFG.ALPHA_VALUES,
         'title': f'CATE MSE vs α ({CFG.PRIMARY_LEARNER})'},
        {'metric': 'kendall_tau', 'alpha_values': [a for a in CFG.ALPHA_VALUES if a > 0],
         'title': f'Kendall τ vs α ({CFG.PRIMARY_LEARNER})'},
    ],
    learner=CFG.PRIMARY_LEARNER,
    save_path=os.path.join(CFG.PLOTS_DIR, 'combined_cate_kendall.png'),
)
plt.show()
